# Section 12: Evaluation, Error Analysis, Explainability, and Go/No-Go
Full evaluation of the trained model from Section 11. Computes per-class metrics, confusion matrix, melanoma-priority analysis, ROC curves, error analysis, and Grad-CAM explainability. Ends with a go/no-go judgment on whether the v1 pipeline is ready or needs revisiting.


In [1]:
%run 01_config.ipynb

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm import tqdm

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, accuracy_score, cohen_kappa_score
)

PREPROCESS_TARGET_SIZE = IMAGE_SIZE[0]  # 224

def preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None):
    raw_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(raw_bytes, channels=3)
    shape = tf.shape(image)
    h = tf.cast(shape[0], tf.float32)
    w = tf.cast(shape[1], tf.float32)
    scale = tf.cast(target_size, tf.float32) / tf.minimum(h, w)
    new_h = tf.cast(tf.math.ceil(h * scale), tf.int32)
    new_w = tf.cast(tf.math.ceil(w * scale), tf.int32)
    image = tf.image.resize(image, [new_h, new_w], method="bilinear")
    image = tf.image.resize_with_crop_or_pad(image, target_size, target_size)
    image = tf.cast(image, tf.float32)
    if preprocess_fn is not None:
        image = preprocess_fn(image)
    else:
        image = image / 255.0
    return image

Creating output structure in: D:\SKIN CANCER/pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.


## Load Test Manifest & Validate


In [2]:
d_splits = os.path.join(OUTPUT_ROOT, "splits")
d_models = os.path.join(OUTPUT_ROOT, "models")
d_eval = os.path.join(OUTPUT_ROOT, "evaluation")
d_explain = os.path.join(OUTPUT_ROOT, "explainability")
os.makedirs(d_eval, exist_ok=True)
os.makedirs(d_explain, exist_ok=True)

df_test = pd.read_csv(os.path.join(d_splits, "test_manifest_cropped.csv"))

expected_test = 2957
expected_test_class = {"NV": 1887, "MEL": 607, "BCC": 463}

print("=== SECTION 12 INPUT VALIDATION ===")
print(f"Test rows: {len(df_test)} (Expected: {expected_test})")
if len(df_test) != expected_test:
    raise ValueError(f"Test manifest row mismatch: expected {expected_test}, got {len(df_test)}")

test_class_counts = df_test["final_authoritative_label"].value_counts()
for cls, exp in expected_test_class.items():
    print(f"  {cls}: {test_class_counts.get(cls, 0)} (Expected: {exp})")

print("\nTest manifest validated.")


=== SECTION 12 INPUT VALIDATION ===
Test rows: 2957 (Expected: 2957)
  NV: 1887 (Expected: 1887)
  MEL: 607 (Expected: 607)
  BCC: 463 (Expected: 463)

Test manifest validated.


## Rebuild Preprocessing (Section 8 + Section 11 Backbone Contract)


In [3]:
def build_test_dataset(df, batch_size=BATCH_SIZE):
    """Deterministic test dataset, no shuffle, no augmentation."""
    paths = df["full_path"].values
    labels = np.array([CLASS_TO_INDEX[l] for l in df["final_authoritative_label"].values])

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_and_backbone_preprocess(path, label):
        image = preprocess_image(path, preprocess_fn=None)
        image = image * 255.0
        image = tf.keras.applications.efficientnet.preprocess_input(image)
        return image, label

    ds = ds.map(load_and_backbone_preprocess, num_parallel_calls=tf.data.AUTOTUNE)   
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

ds_test = build_test_dataset(df_test)
print("Test dataset built.")


Test dataset built.


## Load Best Model from Section 11


In [4]:
best_model_path = os.path.join(d_models, "best_model.keras")
final_model_path = os.path.join(d_models, "final_model.keras")

if os.path.exists(best_model_path):
    model_path_used = best_model_path
    print(f"Loading best model: {best_model_path}")
elif os.path.exists(final_model_path):
    model_path_used = final_model_path
    print(f"Best model not found. Loading final model: {final_model_path}")
else:
    raise FileNotFoundError("No trained model found in pipeline_output/models/.")

model = keras.models.load_model(model_path_used)
print(f"Model loaded. Total params: {model.count_params():,}")


Loading best model: D:\SKIN CANCER/pipeline_output\models\best_model.keras
Model loaded. Total params: 4,213,926


## Generate Predictions on Test Set


In [5]:
print("Running inference on test set...")
y_probs = model.predict(ds_test, verbose=1)
y_pred = np.argmax(y_probs, axis=1)

# Recover true labels in dataset order
y_true = []
for _, labels in ds_test:
    y_true.append(labels.numpy())
y_true = np.concatenate(y_true)

print(f"\nPredictions shape: {y_probs.shape}")
print(f"y_pred shape: {y_pred.shape}")
print(f"y_true shape: {y_true.shape}")

assert len(y_true) == len(y_pred) == len(df_test), "Prediction/label count mismatch"
print("Predictions generated successfully.")


Running inference on test set...
93/93 ━━━━━━━━━━━━━━━━━━━━ 26s 271ms/step

Predictions shape: (2957, 3)
y_pred shape: (2957,)
y_true shape: (2957,)
Predictions generated successfully.


## Overall Metrics


In [6]:
overall_acc = accuracy_score(y_true, y_pred)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
cohen_kappa = cohen_kappa_score(y_true, y_pred)

print("=== OVERALL METRICS ===")
print(f"Accuracy:          {overall_acc:.4f}")
print(f"Macro Precision:   {macro_precision:.4f}")
print(f"Macro Recall:      {macro_recall:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted Precision:{weighted_precision:.4f}")
print(f"Weighted Recall:   {weighted_recall:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Cohen's Kappa:     {cohen_kappa:.4f}")

overall_metrics = pd.DataFrame([{
    "accuracy": overall_acc,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_precision": weighted_precision,
    "weighted_recall": weighted_recall,
    "weighted_f1": weighted_f1,
    "cohen_kappa": cohen_kappa
}])
overall_metrics.to_csv(os.path.join(d_eval, "overall_metrics.csv"), index=False)


=== OVERALL METRICS ===
Accuracy:          0.8028
Macro Precision:   0.7573
Macro Recall:      0.8069
Macro F1:          0.7776
Weighted Precision:0.8203
Weighted Recall:   0.8028
Weighted F1:       0.8078
Cohen's Kappa:     0.6461


## Per-Class Metrics


In [7]:
precision_pc, recall_pc, f1_pc, support_pc = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)

per_class_rows = []
for idx in range(len(CLASS_NAMES)):
    per_class_rows.append({
        "class": INDEX_TO_CLASS[idx],
        "precision": round(float(precision_pc[idx]), 4),
        "recall": round(float(recall_pc[idx]), 4),
        "f1": round(float(f1_pc[idx]), 4),
        "support": int(support_pc[idx])
    })

df_per_class = pd.DataFrame(per_class_rows)
df_per_class.to_csv(os.path.join(d_eval, "per_class_metrics.csv"), index=False)

print("=== PER-CLASS METRICS ===")
display(df_per_class)

# Also save sklearn's classification_report for convenience
report_str = classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4, zero_division=0
)
print("\n=== SKLEARN CLASSIFICATION REPORT ===")
print(report_str)

with open(os.path.join(d_eval, "classification_report.txt"), "w") as f:
    f.write(report_str)


=== PER-CLASS METRICS ===


,class,precision,recall,f1,support
0,NV,0.9055,0.8076,0.8538,1887
1,MEL,0.5863,0.7166,0.6449,607
2,BCC,0.7801,0.8963,0.8342,463



=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

          NV     0.9055    0.8076    0.8538      1887
         MEL     0.5863    0.7166    0.6449       607
         BCC     0.7801    0.8963    0.8342       463

    accuracy                         0.8028      2957
   macro avg     0.7573    0.8069    0.7776      2957
weighted avg     0.8203    0.8028    0.8078      2957



## Confusion Matrix


In [8]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(os.path.join(d_eval, "confusion_matrix.csv"))

print("=== CONFUSION MATRIX (rows=true, cols=predicted) ===")
display(cm_df)

# Normalized (row-normalized to show recall intuitively)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_norm_df.to_csv(os.path.join(d_eval, "confusion_matrix_normalized.csv"))

print("\n=== NORMALIZED CONFUSION MATRIX (row-normalized) ===")
display(cm_norm_df.round(4))

# Plot both
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im1 = ax1.imshow(cm, cmap="Blues", aspect="auto")
ax1.set_title("Confusion Matrix (counts)")
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True")
ax1.set_xticks(range(len(CLASS_NAMES)))
ax1.set_yticks(range(len(CLASS_NAMES)))
ax1.set_xticklabels(CLASS_NAMES)
ax1.set_yticklabels(CLASS_NAMES)
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax1.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im1, ax=ax1)

im2 = ax2.imshow(cm_norm, cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax2.set_title("Confusion Matrix (row-normalized)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("True")
ax2.set_xticks(range(len(CLASS_NAMES)))
ax2.set_yticks(range(len(CLASS_NAMES)))
ax2.set_xticklabels(CLASS_NAMES)
ax2.set_yticklabels(CLASS_NAMES)
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax2.text(j, i, f"{cm_norm[i, j]:.3f}", ha="center", va="center",
                 color="white" if cm_norm[i, j] > 0.5 else "black")
fig.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.savefig(os.path.join(d_eval, "confusion_matrix.png"), dpi=150)
plt.close()
print("\nConfusion matrix plot saved.")


=== CONFUSION MATRIX (rows=true, cols=predicted) ===


,NV,MEL,BCC
NV,1524,280,83
MEL,138,435,34
BCC,21,27,415



=== NORMALIZED CONFUSION MATRIX (row-normalized) ===


,NV,MEL,BCC
NV,0.8076,0.1484,0.0440
MEL,0.2273,0.7166,0.0560
BCC,0.0454,0.0583,0.8963



Confusion matrix plot saved.


## Melanoma-Priority Analysis (Clinical Criticality)


In [9]:
# MEL is the dangerous class. A missed melanoma (false negative) is the most clinically serious error.
# Compute one-vs-rest metrics with MEL as positive class.

mel_idx = CLASS_TO_INDEX["MEL"]
y_true_mel = (y_true == mel_idx).astype(int)
y_pred_mel = (y_pred == mel_idx).astype(int)
y_probs_mel = y_probs[:, mel_idx]

# Counts
tp_mel = int(((y_true_mel == 1) & (y_pred_mel == 1)).sum())
tn_mel = int(((y_true_mel == 0) & (y_pred_mel == 0)).sum())
fp_mel = int(((y_true_mel == 0) & (y_pred_mel == 1)).sum())
fn_mel = int(((y_true_mel == 1) & (y_pred_mel == 0)).sum())

# Clinical metrics
mel_sensitivity = tp_mel / (tp_mel + fn_mel) if (tp_mel + fn_mel) > 0 else 0.0
mel_specificity = tn_mel / (tn_mel + fp_mel) if (tn_mel + fp_mel) > 0 else 0.0
mel_ppv = tp_mel / (tp_mel + fp_mel) if (tp_mel + fp_mel) > 0 else 0.0
mel_npv = tn_mel / (tn_mel + fn_mel) if (tn_mel + fn_mel) > 0 else 0.0
mel_auc = roc_auc_score(y_true_mel, y_probs_mel)

# What did missed melanomas get classified as?
missed_mel_mask = (y_true == mel_idx) & (y_pred != mel_idx)
missed_mel_indices = np.where(missed_mel_mask)[0]
missed_mel_preds = y_pred[missed_mel_indices]
missed_mel_dist = pd.Series(missed_mel_preds).value_counts().to_dict()
missed_mel_dist_named = {INDEX_TO_CLASS[int(k)]: int(v) for k, v in missed_mel_dist.items()}

print("=== MELANOMA-PRIORITY ANALYSIS ===")
print(f"True Positives (MEL correctly detected):    {tp_mel}")
print(f"False Negatives (MISSED melanomas):         {fn_mel}  <-- most dangerous")
print(f"False Positives (overcalled as MEL):        {fp_mel}")
print(f"True Negatives (correctly not MEL):         {tn_mel}")
print(f"")
print(f"MEL Sensitivity (recall):          {mel_sensitivity:.4f}  [% of real MELs caught]")
print(f"MEL Specificity:                   {mel_specificity:.4f}  [% of non-MELs correctly not flagged]")
print(f"MEL Positive Predictive Value:     {mel_ppv:.4f}  [% of MEL predictions that are correct]")
print(f"MEL Negative Predictive Value:     {mel_npv:.4f}  [% of non-MEL predictions that are correct]")
print(f"MEL ROC-AUC:                       {mel_auc:.4f}")
print(f"")
print(f"Missed melanomas classified as: {missed_mel_dist_named}")

mel_analysis = pd.DataFrame([{
    "mel_true_positives": tp_mel,
    "mel_false_negatives": fn_mel,
    "mel_false_positives": fp_mel,
    "mel_true_negatives": tn_mel,
    "mel_sensitivity": round(mel_sensitivity, 4),
    "mel_specificity": round(mel_specificity, 4),
    "mel_ppv": round(mel_ppv, 4),
    "mel_npv": round(mel_npv, 4),
    "mel_auc": round(mel_auc, 4),
    "missed_mel_as_NV": missed_mel_dist_named.get("NV", 0),
    "missed_mel_as_BCC": missed_mel_dist_named.get("BCC", 0)
}])
mel_analysis.to_csv(os.path.join(d_eval, "melanoma_priority_analysis.csv"), index=False)


=== MELANOMA-PRIORITY ANALYSIS ===
True Positives (MEL correctly detected):    435
False Negatives (MISSED melanomas):         172  <-- most dangerous
False Positives (overcalled as MEL):        307
True Negatives (correctly not MEL):         2043

MEL Sensitivity (recall):          0.7166  [% of real MELs caught]
MEL Specificity:                   0.8694  [% of non-MELs correctly not flagged]
MEL Positive Predictive Value:     0.5863  [% of MEL predictions that are correct]
MEL Negative Predictive Value:     0.9223  [% of non-MEL predictions that are correct]
MEL ROC-AUC:                       0.8925

Missed melanomas classified as: {'NV': 138, 'BCC': 34}


## Per-Class ROC-AUC (One-vs-Rest)


In [10]:
print("=== ONE-VS-REST ROC-AUC ===")

fig, ax = plt.subplots(1, 1, figsize=(8, 7))

auc_rows = []
for idx in range(len(CLASS_NAMES)):
    cls = INDEX_TO_CLASS[idx]
    y_binary = (y_true == idx).astype(int)
    y_score = y_probs[:, idx]

    auc = roc_auc_score(y_binary, y_score)
    fpr, tpr, _ = roc_curve(y_binary, y_score)

    ax.plot(fpr, tpr, label=f"{cls} (AUC = {auc:.4f})", linewidth=2)
    auc_rows.append({"class": cls, "roc_auc": round(float(auc), 4)})
    print(f"  {cls}: AUC = {auc:.4f}")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("One-vs-Rest ROC Curves")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(os.path.join(d_eval, "roc_curves.png"), dpi=150)
plt.close()

pd.DataFrame(auc_rows).to_csv(os.path.join(d_eval, "per_class_auc.csv"), index=False)
print("\nROC curves saved.")


=== ONE-VS-REST ROC-AUC ===
  NV: AUC = 0.9174
  MEL: AUC = 0.8925
  BCC: AUC = 0.9834

ROC curves saved.


## Error Analysis: Highest-Confidence Mistakes


In [11]:
# Build a per-sample error dataframe
error_df = df_test.copy().reset_index(drop=True)
error_df["true_label_idx"] = y_true
error_df["pred_label_idx"] = y_pred
error_df["pred_label"] = [INDEX_TO_CLASS[int(i)] for i in y_pred]
error_df["is_correct"] = (y_true == y_pred)
error_df["prob_NV"] = y_probs[:, 0]
error_df["prob_MEL"] = y_probs[:, 1]
error_df["prob_BCC"] = y_probs[:, 2]
error_df["pred_confidence"] = y_probs[np.arange(len(y_probs)), y_pred]

errors_only = error_df[~error_df["is_correct"]].copy()
errors_only = errors_only.sort_values("pred_confidence", ascending=False)

error_cols = ["full_path", "final_authoritative_label", "pred_label",
              "prob_NV", "prob_MEL", "prob_BCC", "pred_confidence"]
errors_only[error_cols].to_csv(os.path.join(d_eval, "errors_ranked_by_confidence.csv"), index=False)

# Save the full prediction table too
error_df[["full_path", "final_authoritative_label", "pred_label", "is_correct",
          "prob_NV", "prob_MEL", "prob_BCC", "pred_confidence"]].to_csv(
    os.path.join(d_eval, "all_test_predictions.csv"), index=False
)

print(f"=== ERROR SUMMARY ===")
print(f"Total errors: {len(errors_only)} / {len(error_df)} ({100*len(errors_only)/len(error_df):.2f}%)")
print(f"")
print("Error breakdown (true -> predicted):")
error_breakdown = errors_only.groupby(["final_authoritative_label", "pred_label"]).size().reset_index(name="count")
display(error_breakdown.sort_values("count", ascending=False))

print("\nTop 10 highest-confidence mistakes:")
display(errors_only[error_cols].head(10))


=== ERROR SUMMARY ===
Total errors: 583 / 2957 (19.72%)

Error breakdown (true -> predicted):


,final_authoritative_label,pred_label,count
5,NV,MEL,280
3,MEL,NV,138
4,NV,BCC,83
2,MEL,BCC,34
0,BCC,MEL,27
1,BCC,NV,21



Top 10 highest-confidence mistakes:


,full_path,final_authoritative_label,pred_label,prob_NV,prob_MEL,prob_BCC,pred_confidence
1582,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,BCC,0.002164,0.000059,9.977768e-01,0.997777
2024,D:\SKIN CANCER/pipeline_output\cropped_images\...,MEL,NV,0.996991,0.003006,3.452426e-06,0.996991
1861,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,MEL,0.003294,0.996706,4.751896e-08,0.996706
1659,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,BCC,0.003222,0.000113,9.966653e-01,0.996665
1381,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,BCC,0.001186,0.003519,9.952952e-01,0.995295
1832,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,MEL,0.003562,0.994501,1.937043e-03,0.994501
1742,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,MEL,0.005584,0.994414,1.657962e-06,0.994414
1477,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,BCC,0.003971,0.001648,9.943809e-01,0.994381
1545,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,BCC,0.005651,0.000369,9.939803e-01,0.993980
470,D:\SKIN CANCER/pipeline_output\cropped_images\...,NV,BCC,0.004005,0.002305,9.936900e-01,0.993690


## Visualize High-Confidence Errors


In [12]:
import cv2

def plot_error_grid(err_df, title, save_path, n=9):
    """Display n highest-confidence errors with true/pred labels."""
    n = min(n, len(err_df))
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 12))
    fig.suptitle(title, fontsize=14)

    for i in range(n):
        ax = axes[i // cols][i % cols] if rows > 1 else axes[i % cols]
        row = err_df.iloc[i]
        img = cv2.imread(row["full_path"])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        title_str = f"True: {row['final_authoritative_label']} -> Pred: {row['pred_label']}\nconf={row['pred_confidence']:.3f}"
        ax.set_title(title_str, fontsize=9)
        ax.axis("off")

    for i in range(n, rows * cols):
        ax = axes[i // cols][i % cols] if rows > 1 else axes[i % cols]
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved: {save_path}")


# Top errors overall
plot_error_grid(errors_only, "Top Errors by Confidence",
                os.path.join(d_eval, "errors_top_by_confidence.png"), n=9)

# Missed melanomas specifically (clinically most important)
missed_mel_df = error_df[(error_df["final_authoritative_label"] == "MEL") &
                          (error_df["pred_label"] != "MEL")].copy()
missed_mel_df = missed_mel_df.sort_values("pred_confidence", ascending=False)

if len(missed_mel_df) > 0:
    plot_error_grid(missed_mel_df, "Missed Melanomas (False Negatives) by Confidence",
                    os.path.join(d_eval, "errors_missed_melanomas.png"), n=9)
    missed_mel_df[error_cols].to_csv(os.path.join(d_eval, "missed_melanomas.csv"), index=False)
else:
    print("No missed melanomas to visualize (or all were caught).")


Saved: D:\SKIN CANCER/pipeline_output\evaluation\errors_top_by_confidence.png
Saved: D:\SKIN CANCER/pipeline_output\evaluation\errors_missed_melanomas.png


## Grad-CAM Explainability


In [13]:
# Find the last Conv2D layer inside the EfficientNetB0 backbone for Grad-CAM
def find_last_conv_layer(model):
    """Find last conv layer, searching recursively into nested models."""
    # First check if there's a nested model (the EfficientNet backbone)
    for layer in reversed(model.layers):
        if isinstance(layer, keras.Model):
            # Found the backbone model; search inside it
            for sub_layer in reversed(layer.layers):
                if isinstance(sub_layer, keras.layers.Conv2D):
                    return layer.name, sub_layer.name
    # Fallback: search top-level
    for layer in reversed(model.layers):
        if isinstance(layer, keras.layers.Conv2D):
            return None, layer.name
    return None, None

backbone_name, last_conv_name = find_last_conv_layer(model)
print(f"Backbone (nested model): {backbone_name}")
print(f"Last conv layer: {last_conv_name}")

# Build a Grad-CAM model. Since the backbone is nested, we need to reach inside it.
if backbone_name is not None:
    backbone = model.get_layer(backbone_name)
    last_conv_layer = backbone.get_layer(last_conv_name)

    # Build a functional model from the original input all the way through
    # The trick: run the backbone up to last_conv, then continue through the head.
    grad_backbone = keras.Model(
        inputs=backbone.input,
        outputs=[last_conv_layer.output, backbone.output]
    )
else:
    grad_backbone = None
    print("Could not find nested backbone; Grad-CAM may not work correctly.")


Backbone (nested model): efficientnetb0
Last conv layer: top_conv


In [14]:
def compute_gradcam(image_tensor, class_idx):
    """
    Compute Grad-CAM heatmap for a single preprocessed image tensor.

    Args:
        image_tensor: (224, 224, 3) float32, already preprocessed.
        class_idx: int, target class for gradient.

    Returns:
        heatmap of shape (H, W), values in [0, 1].
    """
    img_batch = tf.expand_dims(image_tensor, axis=0)

    # Build the gradient tape through both backbone and head
    with tf.GradientTape() as tape:
        # Forward pass through backbone, capturing last conv output
        conv_output, backbone_feat = grad_backbone(img_batch)
        tape.watch(conv_output)

        # Continue through the head layers of the top-level model
        # Head layers are: GAP -> Dropout -> Dense -> Dropout -> Dense(softmax)
        x = backbone_feat
        # Find the head layers (everything after the backbone in the main model)
        backbone_done = False
        for layer in model.layers:
            if layer.name == backbone_name:
                backbone_done = True
                continue
            if backbone_done and not isinstance(layer, keras.layers.InputLayer):
                x = layer(x, training=False) if "training" in layer.call.__code__.co_varnames else layer(x)
        predictions = x
        class_score = predictions[:, class_idx]

    # Compute gradient of class score w.r.t. last conv feature map
    grads = tape.gradient(class_score, conv_output)
    if grads is None:
        return np.zeros((PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE))

    # Global average pooling of gradients -> channel importance weights
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the conv output by importance and sum over channels
    conv_output = conv_output[0]  # remove batch dim
    heatmap = tf.reduce_sum(conv_output * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)  # ReLU

    # Normalize
    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val

    heatmap = heatmap.numpy()

    # Upsample to 224x224
    heatmap = cv2.resize(heatmap, (PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE))

    return heatmap


def overlay_heatmap(image_raw_path, heatmap, target_size=PREPROCESS_TARGET_SIZE, alpha=0.4):
    """Overlay Grad-CAM heatmap on the original image."""
    img = cv2.imread(image_raw_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Match the same resize-and-center-crop as preprocessing
    h, w = img.shape[:2]
    scale = target_size / min(h, w)
    new_h = int(np.ceil(h * scale))
    new_w = int(np.ceil(w * scale))
    img = cv2.resize(img, (new_w, new_h))
    start_y = (new_h - target_size) // 2
    start_x = (new_w - target_size) // 2
    img = img[start_y:start_y+target_size, start_x:start_x+target_size]

    # Colorize heatmap
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    # Overlay
    overlay = np.uint8(img * (1 - alpha) + heatmap_colored * alpha)
    return img, overlay

print("Grad-CAM helpers defined.")


Grad-CAM helpers defined.


In [15]:
# Generate Grad-CAM for a sample of correct and incorrect predictions per class
def generate_gradcam_samples(error_df, n_per_class=3):
    """Generate Grad-CAM overlays for correct and incorrect predictions."""
    samples = []
    for cls in CLASS_NAMES:
        cls_idx = CLASS_TO_INDEX[cls]
        correct = error_df[(error_df["final_authoritative_label"] == cls) & error_df["is_correct"]]
        incorrect = error_df[(error_df["final_authoritative_label"] == cls) & ~error_df["is_correct"]]

        for _, row in correct.sample(min(n_per_class, len(correct)), random_state=RANDOM_SEED).iterrows():
            samples.append((row, cls_idx, "correct"))
        for _, row in incorrect.sample(min(n_per_class, len(incorrect)), random_state=RANDOM_SEED).iterrows():
            samples.append((row, cls_idx, "incorrect"))
    return samples


samples = generate_gradcam_samples(error_df, n_per_class=3)
print(f"Generating Grad-CAM for {len(samples)} samples...")

fig, axes = plt.subplots(len(samples), 2, figsize=(8, 3 * len(samples)))
fig.suptitle("Grad-CAM: Model Attention", fontsize=14)

for i, (row, cls_idx, status) in enumerate(tqdm(samples, desc="Grad-CAM")):
    image_tensor = preprocess_image(row["full_path"])
    heatmap = compute_gradcam(image_tensor, cls_idx)
    orig, overlay = overlay_heatmap(row["full_path"], heatmap)

    ax_orig = axes[i][0]
    ax_over = axes[i][1]
    ax_orig.imshow(orig)
    ax_orig.set_title(f"{row['final_authoritative_label']} ({status}) -> {row['pred_label']}", fontsize=9)
    ax_orig.axis("off")
    ax_over.imshow(overlay)
    ax_over.set_title(f"Grad-CAM for {INDEX_TO_CLASS[cls_idx]}", fontsize=9)
    ax_over.axis("off")

plt.tight_layout()
gradcam_path = os.path.join(d_explain, "gradcam_samples.png")
plt.savefig(gradcam_path, dpi=150)
plt.close()
print(f"Grad-CAM saved: {gradcam_path}")


Generating Grad-CAM for 18 samples...


Grad-CAM: 100%|██████████| 18/18 [00:06<00:00,  2.78it/s]


Grad-CAM saved: D:\SKIN CANCER/pipeline_output\explainability\gradcam_samples.png


## Go / No-Go Judgment


In [16]:
# Define clinically-informed thresholds for a v1 baseline
# These are baseline targets, not clinical validation thresholds
GO_NOGO_THRESHOLDS = {
    "overall_accuracy_min": 0.75,
    "mel_sensitivity_min": 0.70,    # Must catch at least 70% of melanomas
    "mel_auc_min": 0.80,
    "macro_f1_min": 0.65
}

criteria = [
    {"criterion": "overall_accuracy",    "value": overall_acc,         "threshold": GO_NOGO_THRESHOLDS["overall_accuracy_min"],  "pass": overall_acc >= GO_NOGO_THRESHOLDS["overall_accuracy_min"]},
    {"criterion": "mel_sensitivity",     "value": mel_sensitivity,     "threshold": GO_NOGO_THRESHOLDS["mel_sensitivity_min"],   "pass": mel_sensitivity >= GO_NOGO_THRESHOLDS["mel_sensitivity_min"]},
    {"criterion": "mel_auc",             "value": mel_auc,             "threshold": GO_NOGO_THRESHOLDS["mel_auc_min"],           "pass": mel_auc >= GO_NOGO_THRESHOLDS["mel_auc_min"]},
    {"criterion": "macro_f1",            "value": macro_f1,            "threshold": GO_NOGO_THRESHOLDS["macro_f1_min"],          "pass": macro_f1 >= GO_NOGO_THRESHOLDS["macro_f1_min"]}
]

df_go_nogo = pd.DataFrame(criteria)
df_go_nogo["value"] = df_go_nogo["value"].round(4)
df_go_nogo.to_csv(os.path.join(d_eval, "go_nogo_criteria.csv"), index=False)

print("=== GO / NO-GO JUDGMENT ===")
display(df_go_nogo)

all_pass = df_go_nogo["pass"].all()
any_pass = df_go_nogo["pass"].any()

if all_pass:
    verdict = "GO — all baseline criteria met. Proceed with deployment decisions cautiously."
elif any_pass:
    verdict = "PARTIAL — some criteria met. Targeted improvements recommended before any clinical use."
else:
    verdict = "NO-GO — no baseline criteria met. Substantial revisions needed."

print(f"\nVerdict: {verdict}")
print("\nIMPORTANT: These are engineering baseline thresholds only. This does NOT constitute clinical validation.")


=== GO / NO-GO JUDGMENT ===


,criterion,value,threshold,pass
0,overall_accuracy,0.8028,0.75,True
1,mel_sensitivity,0.7166,0.70,True
2,mel_auc,0.8925,0.80,True
3,macro_f1,0.7776,0.65,True



Verdict: GO — all baseline criteria met. Proceed with deployment decisions cautiously.

IMPORTANT: These are engineering baseline thresholds only. This does NOT constitute clinical validation.


## Section 12 Summary


In [17]:
print("=== SECTION 12 FINAL SUMMARY ===")
print(f"Model evaluated: {os.path.basename(model_path_used)}")
print(f"Test samples: {len(error_df)}")
print(f"")
print(f"Overall accuracy: {overall_acc:.4f}")
print(f"Macro F1:         {macro_f1:.4f}")
print(f"Cohen's Kappa:    {cohen_kappa:.4f}")
print(f"")
print(f"Per-class recall:")
for idx in range(len(CLASS_NAMES)):
    print(f"  {INDEX_TO_CLASS[idx]}: {recall_pc[idx]:.4f} (support={int(support_pc[idx])})")
print(f"")
print(f"MEL sensitivity: {mel_sensitivity:.4f}  (missed {fn_mel} of {tp_mel+fn_mel} real melanomas)")
print(f"MEL specificity: {mel_specificity:.4f}")
print(f"MEL AUC:         {mel_auc:.4f}")
print(f"")
print(f"Verdict: {verdict}")

print("\nSaved files:")
print("  evaluation/")
print("  - overall_metrics.csv")
print("  - per_class_metrics.csv")
print("  - classification_report.txt")
print("  - confusion_matrix.csv")
print("  - confusion_matrix_normalized.csv")
print("  - confusion_matrix.png")
print("  - melanoma_priority_analysis.csv")
print("  - per_class_auc.csv")
print("  - roc_curves.png")
print("  - all_test_predictions.csv")
print("  - errors_ranked_by_confidence.csv")
print("  - errors_top_by_confidence.png")
print("  - errors_missed_melanomas.png (if applicable)")
print("  - missed_melanomas.csv (if applicable)")
print("  - go_nogo_criteria.csv")
print("  explainability/")
print("  - gradcam_samples.png")

print("\nSection 12 completed. Next step: review outputs and decide which enhancements to prioritize.")


=== SECTION 12 FINAL SUMMARY ===
Model evaluated: best_model.keras
Test samples: 2957

Overall accuracy: 0.8028
Macro F1:         0.7776
Cohen's Kappa:    0.6461

Per-class recall:
  NV: 0.8076 (support=1887)
  MEL: 0.7166 (support=607)
  BCC: 0.8963 (support=463)

MEL sensitivity: 0.7166  (missed 172 of 607 real melanomas)
MEL specificity: 0.8694
MEL AUC:         0.8925

Verdict: GO — all baseline criteria met. Proceed with deployment decisions cautiously.

Saved files:
  evaluation/
  - overall_metrics.csv
  - per_class_metrics.csv
  - classification_report.txt
  - confusion_matrix.csv
  - confusion_matrix_normalized.csv
  - confusion_matrix.png
  - melanoma_priority_analysis.csv
  - per_class_auc.csv
  - roc_curves.png
  - all_test_predictions.csv
  - errors_ranked_by_confidence.csv
  - errors_top_by_confidence.png
  - errors_missed_melanomas.png (if applicable)
  - missed_melanomas.csv (if applicable)
  - go_nogo_criteria.csv
  explainability/
  - gradcam_samples.png

Section 12 co